# Calculate error rates from Sep/3-Sep/8 in office testing. 
Testing build Survey 2026 build #36 d6ced9e. Built on 2026-09-03T02:40:01Z

In [30]:
import json

In [31]:
#define the path to the reference text
reference_file = "short_swahili_answers.json"
reference = reference_file

# Read from a relative path
with open(reference, 'r') as g:
    ref_data = json.load(g)

print (f"Reference data: loaded {len(ref_data)} entries from {reference_file}")
print(json.dumps(ref_data, indent=4, ensure_ascii=False))

Reference data: loaded 10 entries from short_swahili_answers.json
{
    "Q8": "Ndiyo, iliharibu majani na kupunguza mavuno.",
    "Q9": "Naweza kukubali kupoteza hadi 10% ya mavuno.",
    "Q10": "Ikiwa uharibifu unafika karibu 30%, nitabadilisha aina.",
    "Q11": "Iwe na mavuno mengi na iwe inastahimili ukame na wadudu.",
    "Q12": "Hata tani 1 kwa ekari, kama mbegu ni nzuri.",
    "Q13": "Wakati wa maua na kutengeneza punje.",
    "Q14": "Ninaweka ya chakula, nauza iliyobaki na kidogo ni ya mifugo.",
    "Q15": "Nawapa kuku na nguruwe mahindi mara mbili hadi tatu kwa wiki.",
    "Q16": "Ninanunua kwa agrovet kwa sababu ninaamini ubora wake.",
    "Q17": "Nauza kwa wafanyabiashara wa eneo kwa sababu wako karibu na hulipa haraka."
}


In [32]:
# Define the GitHub URLs of the surveys we want to test today.
# Paste the normal github.com "blob" links; they are converted to raw file URLs below.
survey_urls = [
    "https://github.com/ishizuki-tech/SurveyExports/blob/main/2026-09-03/exports/survey_1507f113-a21a-4d59-9715-fc0d9c3c1272_2026-09-03_12-19-28.json",
    "https://github.com/ishizuki-tech/SurveyExports/blob/main/2026-09-04/exports/survey_490d0918-ac6d-4df6-bc82-862e756a7537_2026-09-04_13-25-02.json",
    "https://github.com/ishizuki-tech/SurveyExports/blob/main/2026-09-07/exports/survey_2b55924d-c477-442a-85d7-6a84304746bd_2026-09-07_11-12-07.json",
    "https://github.com/ishizuki-tech/SurveyExports/blob/main/2026-09-08/exports/survey_579d0196-d0a2-47aa-a746-5d17336550c4_2026-09-08_15-30-07.json",
]

from urllib.request import urlopen

# One entry per survey: file name plus the reference/answer pairs that matched.
surveys = []
for survey_url in survey_urls:
    raw_url = survey_url.replace("https://github.com/", "https://raw.githubusercontent.com/", 1).replace("/blob/", "/", 1)
    with urlopen(raw_url) as f:
        survey_data = json.load(f)

    survey_file = raw_url.rsplit("/", 1)[-1]
    print(f"\nLoaded data from {survey_file}")

    # Keep only the questions that have a reference answer, in reference order.
    questions = [q for q in ref_data if q in survey_data['answers']]
    missing = [q for q in ref_data if q not in survey_data['answers']]
    if missing:
        print(f"Not in survey, skipped: {missing}")

    references = [ref_data[q] for q in questions]
    answers = [survey_data['answers'][q]['answer'] for q in questions]
    print(f"Found {len(answers)} transcribed answers matching the reference.")
    for q, ans in zip(questions, answers):
        print(f"{q}: {ans}")

    surveys.append({"file": survey_file, "references": references, "answers": answers})


Loaded data from survey_1507f113-a21a-4d59-9715-fc0d9c3c1272_2026-09-03_12-19-28.json
Found 10 transcribed answers matching the reference.
Q8: Ndiyo iliharibu majani na kupunguza mawuno.
Q9: Nwiza kukubali kupoteza hadi asili miakumi yamavuno.
Q10: Ikiva uhare bifu unafika karibu asilimia telatini mitaba dilisha aina.
Q11: Iwa na mawunomegi na iwa inastahi mili ukame na wa dulu.
Q12: Hatata ni moja kwaikari, kamambeguninzuri.
Q13: Wakati wamaua na kutengeneza punje
Q14: Ninaweka ya chakula, nauza iliobaki na kidogo ni yami fugo.
Q15: Nawa paku kuna gurue mahindi ma rambili hadita tukwa wiki.
Q16: Nina nunuwa kwa agrovet kwa sababu Nina mini uborawake.
Q17: Nauza kwa wafani biesha rawa eneo kwa sababu wa koka ribu na huli paharaka.

Loaded data from survey_490d0918-ac6d-4df6-bc82-862e756a7537_2026-09-04_13-25-02.json
Found 10 transcribed answers matching the reference.
Q8: na una prashto wa?
 dia, ilha rebo maja ni na kupunguza mabuno.
Q9: Niza kukubare kupoteza hadyesrimi akumia mawun

In [33]:
import jiwer 

transform_words = jiwer.Compose ([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(), 
    jiwer.ReduceToListOfListOfWords()
])

transform_chars = jiwer.Compose ([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(), 
    jiwer.ReduceToListOfListOfChars()
])

def error_rates(references, answers):
    out = jiwer.process_words(references, answers, reference_transform=transform_words, hypothesis_transform=transform_words)
    out2 = jiwer.process_characters(references, answers, reference_transform=transform_chars, hypothesis_transform=transform_chars)
    return out.wer, out2.cer

# Per-survey rates.
for s in surveys:
    wer, cer = error_rates(s["references"], s["answers"])
    print(f"{s['file']}  ({len(s['answers'])} answers)")
    print(f"  WER (normalized): {wer:.3f}   CER (normalized): {cer:.3f}")

# Aggregate over every answer from every survey: total errors / total reference words (or characters).
all_references = [r for s in surveys for r in s["references"]]
all_answers = [a for s in surveys for a in s["answers"]]
wer, cer = error_rates(all_references, all_answers)
print(f"\nAggregate over {len(surveys)} surveys, {len(all_answers)} answers")
print(f"  WER (normalized): {wer:.3f}   CER (normalized): {cer:.3f}")

survey_1507f113-a21a-4d59-9715-fc0d9c3c1272_2026-09-03_12-19-28.json  (10 answers)
  WER (normalized): 0.651   CER (normalized): 0.166
survey_490d0918-ac6d-4df6-bc82-862e756a7537_2026-09-04_13-25-02.json  (10 answers)
  WER (normalized): 0.837   CER (normalized): 0.286
survey_2b55924d-c477-442a-85d7-6a84304746bd_2026-09-07_11-12-07.json  (10 answers)
  WER (normalized): 0.733   CER (normalized): 0.176
survey_579d0196-d0a2-47aa-a746-5d17336550c4_2026-09-08_15-30-07.json  (10 answers)
  WER (normalized): 0.767   CER (normalized): 0.231

Aggregate over 4 surveys, 40 answers
  WER (normalized): 0.747   CER (normalized): 0.215


## Extra speech in the transcriptions

Three of the 40 scored answers contain extra speech: a separate first line, followed by the actual answer. The other 37 answers start and end where the reference does; their errors are misheard or split words, not extra speech.

| Survey | Question | Extra speech | Answer that follows |
|---|---|---|---|
| 2026-09-04 (`490d0918`) | Q8 | "na una prashto wa?" | "dia, ilha rebo maja ni na kupunguza mabuno." |
| 2026-09-04 (`490d0918`) | Q12 | "So mreka wakotisha mifika mwisha." | "Hata tani moja kwaikari kama beguni zuri." |
| 2026-09-08 (`579d0196`) | Q8 | "Okay." | "Dio niri, niri haribu maja ni na kupungu zamabuno." |

- **2026-09-04, Q8:** "na una prashto wa?" is a question. It sounds like the interviewer or someone nearby speaking before the respondent answered.
- **2026-09-04, Q12:** the extra sentence is as long as the answer, so it's probably a whole other remark caught before the respondent spoke.
- **2026-09-08, Q8:** "Okay." at the very start of the first scored question sounds like the interviewer starting the recording.

This is judged from the text only. Listen to the audio to confirm who was speaking.

### Effect on the error rates

Scored with the three extra-speech lines removed (the calculation above keeps them):

| | WER | CER |
|---|---|---|
| 2026-09-04 as recorded | 0.837 | 0.286 |
| 2026-09-04 without the extra speech | 0.733 | 0.192 |
| 2026-09-08 as recorded | 0.767 | 0.231 |
| 2026-09-08 without the extra speech | 0.756 | 0.223 |
| **Aggregate as recorded** | **0.747** | **0.215** |
| **Aggregate without the extra speech** | **0.718** | **0.189** |